In [1]:
"""
A Gradio demo web application developed for Cardiac MRI segmentation.
"""

# Import libraries
import random
import gradio as gr
import torch
import numpy as np
import albumentations as A
from pathlib import Path
from PIL import Image
from src.org_unet import UNet
from src.residual_unet import ResidualUNet
from src.attention_unet import AttentionUNetV3
from src.feature_pyramid_unet import FeaturePyramidUNet
from src.feedback_resunet import FeedbackResUNet
from src.transUnet import TransformerUNet

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class_colors = {
    0: (0, 0, 0),        # Background - Black
    1: (255, 0, 0),      # Class 1 - Red
    2: (0, 255, 0),      # Class 2 - Green
    3: (0, 0, 255),      # Class 3 - Blue
}


def load_model(checkpoint_path, model_class):
    """
    Load a model from a given checkpoint.

    Parameters:
        checkpoint_path (Path): Path to the model checkpoint.
        model_class: Class of the model architecture.

    Returns:
        torch.nn.Module: Loaded model.
    """
    model_ = model_class()
    checkpoint = torch.load(checkpoint_path)
    model_.load_state_dict(checkpoint)
    model_ = model_.eval()
    return model_


def preprocess_image(image):
    """
    Preprocess the image for segmentation.
    Parameters:
        image (np.array): Image to be preprocessed.
    Returns:
        image (np.array): Preprocessed image.
    """
    image = np.array(image.resize((224, 224))).astype(np.float32) / 255.0  # [0,1]
    if image.ndim == 3:
        image = image[..., 0]  # Convert RGB to grayscale if needed

    image = (image * 255).astype(np.uint8)  # Scale to [0, 255]

    # Normalize using Albumentations
    normalize_transform = A.Normalize(mean=0.5, std=0.5, max_pixel_value=1.0)
    normalized = normalize_transform(image=image)
    image = torch.from_numpy(normalized["image"]).unsqueeze(0).unsqueeze(0)  #Add batch and channel dimensions [1, 1, H, W]

    return image.float()


def apply_colormap(mask):
    """Convert class mask to RGB image."""
    h, w = mask.shape
    rgb_mask = np.zeros((h, w, 3), dtype=np.uint8)
    for cls, color in class_colors.items():
        rgb_mask[mask == cls] = color
    return Image.fromarray(rgb_mask)


def segment_cmri(image):
    """
    Segment the image using a trained model.
    Parameters:
        image (np.array): Image to be segmented.
    Returns:
        results (np.array): Segmented image.
    """
    # Load model
    model_path = Path(f'./models/checkpoints/feat_pyramid_unet_img_slices_with_ratios_v4_20241224021156-rumbling-dog-720/checkpoint_epoch{23}.pth')
    model = load_model(checkpoint_path=model_path, model_class=FeaturePyramidUNet)

    # Preprocess the image
    image = preprocess_image(image)

    with torch.no_grad():
        image = image.to(device, dtype=torch.float32, memory_format=torch.channels_last)

        model = model.to(device)

        prediction = model(image)  # Get model prediction
        prediction = torch.argmax(prediction, dim=1).squeeze(0).cpu().numpy()

        colored_mask = apply_colormap(prediction)

    return colored_mask


# Gradio Demo
demo = gr.Interface(
    fn=segment_cmri,
    inputs=gr.Image(type="pil", image_mode='L', label="Input Image"),
    outputs=gr.Image(type="pil", label="Segmented Image"),
    title="CMRI Segmentation Demo",
    examples=[["../data/ACDC/img_slices_with_ratios_v4/testing/labeled_-1/images/subject136_frame01_slice01_ras.png"]]
)

demo.launch(share=False, debug=True)

C:\Users\Chinthalanka\anaconda3\envs\pytorch_gpu\lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,
INFO:albumentations.check_version:A new version of Albumentations is available: 2.0.5 (you have 1.4.7). Upgrade using: pip install --upgrade albumentations


Running on local URL:  http://127.0.0.1:7860


INFO:httpx:HTTP Request: GET https://checkip.amazonaws.com/ "HTTP/1.1 200 "
INFO:httpx:HTTP Request: GET http://127.0.0.1:7860/startup-events "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"



To create a public link, set `share=True` in `launch()`.


INFO:httpx:HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"


Keyboard interruption in main thread... closing server.
